# TB Portals - Kantipudi **A2** baseline (ALP regressor + cavity classifier)

Locked config: **whole-image cavity** (`--cavity-no-lung-crop`) + **cropped ALP**. 3 held-out countries x 3 seeds x 30 epochs. **Attach these Kaggle datasets before running:** `tb-portals-cxr-pngs`, `medsam-vit-b`.

## 0 - Clone the codebase

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/mabdullahi7780/dl-project-codebase.git"
REPO_DIR = "/kaggle/working/dl-project-codebase"
BRANCH   = "cleaned-repo"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
for _p in (REPO_DIR, REPO_DIR + '/scripts'):
    if _p not in sys.path:
        sys.path.insert(0, _p)
print("repo ready at", REPO_DIR)

## Install deps

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "segment-anything", "pydicom", "pylibjpeg", "pylibjpeg-libjpeg"], check=False)
print("deps installed")

## Paths

Edit dataset slugs if yours differ.

In [ ]:
import os
WORK           = "/kaggle/working"
REPO_DIR       = "/kaggle/working/dl-project-codebase"
DATASET        = "/kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs"
KAGGLE_EXPORT  = f"{DATASET}/kaggle_export"
MEDSAM_CKPT    = "/kaggle/input/datasets/iahmedhabib/medsam-vit-b/medsam_vit_b.pth"
LUNG_DECODER   = f"{REPO_DIR}/checkpoints/component4/component4_mask_decoder.pt"
PAPER_MANIFEST = f"{WORK}/tbportals_manifest_paper.csv"
CROPS_DIR      = f"{WORK}/crops"
OUT_DIR        = f"{WORK}/checkpoints/paper_a2"
os.makedirs(OUT_DIR, exist_ok=True)
print("KAGGLE_EXPORT:", KAGGLE_EXPORT, "->", os.path.isdir(KAGGLE_EXPORT))
print("MEDSAM_CKPT:  ", MEDSAM_CKPT, "->", os.path.isfile(MEDSAM_CKPT))
print("LUNG_DECODER: ", LUNG_DECODER, "->", os.path.isfile(LUNG_DECODER))

## 1 - Build the 5,010-image manifest (Kantipudi Table 1)

In [ ]:
import sys, pandas as pd
from pathlib import Path
if REPO_DIR + '/scripts' not in sys.path:
    sys.path.insert(0, REPO_DIR + '/scripts')
from build_paper_manifest import subsample, PAPER_TOTAL
raw = pd.read_csv(f"{KAGGLE_EXPORT}/manifest.csv",
                  dtype={"image_id": str, "patient_id": str, "country": str})
raw["image_path"] = raw["image_path"].apply(
    lambda p: p if str(p).startswith("/") else f"{KAGGLE_EXPORT}/{p}")
paper_df = subsample(raw, seed=42)
paper_df["image_id"] = paper_df["image_path"].apply(lambda p: Path(str(p)).stem)
paper_df.to_csv(PAPER_MANIFEST, index=False)
print(f"Paper manifest: {len(paper_df)} images (target {PAPER_TOTAL}) -> {PAPER_MANIFEST}")

## 2 - MedSAM lung crops (~25 min first time; idempotent)

In [ ]:
import os, sys
if REPO_DIR + '/scripts' not in sys.path:
    sys.path.insert(0, REPO_DIR + '/scripts')
from cache_lung_crops import main as crops_main
argv = ['--manifest', PAPER_MANIFEST, '--out-dir', CROPS_DIR,
        '--medsam-ckpt', MEDSAM_CKPT, '--size', '224', '--pad', '32']
if os.path.isfile(LUNG_DECODER):
    argv += ['--lung-decoder-ckpt', LUNG_DECODER]
crops_main(argv)
print('crops ->', CROPS_DIR, '| count:', len(os.listdir(CROPS_DIR)))

## 3 - Full A2 run (~2-3 h)

In [ ]:
from src.training.train_baseline_paper import main as train_main
train_main(['--manifest', PAPER_MANIFEST, '--crops-dir', CROPS_DIR, '--out-dir', OUT_DIR,
            '--held-outs','Romania','Moldova','Kazakhstan','--seeds','0','1','2',
            '--epochs','30','--batch-size','60','--accum-steps','5','--num-workers','2',
            '--cavity-no-lung-crop'])

## 4 - Save outputs (download these)

In [ ]:
!cd /kaggle/working && zip -j results_a2.zip checkpoints/paper_a2/results.csv tbportals_manifest_paper.csv
!cd /kaggle/working && zip -r -q checkpoints_a2.zip checkpoints/paper_a2
print("Saved: results_a2.zip, checkpoints_a2.zip")